<div style="font-size: 1em; display: flex; align-items: center; gap: 8px; padding: 8px 16px; background: #F8F9FA; border-bottom: 2px solid #E0E0E0; margin: 0; line-height: 1">
    <img src="https://cdn.simpleicons.org/databricks/FF3621" width="24" height="24"/>
    <div style="color: #666">
        <span style="font-weight: bold; color: #333">Databricks Performance Optimization
</span>
        <span style="margin-left: 8px; color: #999">|</span>
        <span style="margin-left: 8px">05 Demo - Caching and Intermediate Results</span>
    </div>
</div>

![DB Academy](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/databricks_academy_logo.png)

# Demo - Caching and Intermediate Results
#### Duration: 15 minutes

## Overview

A DataFrame is a logical plan, not data held in memory, so each action re-reads from storage unless something has cached the result. Understanding when repeat access happens is the key to understanding caching.

Several caches sit between your query and cloud storage, and the platform manages almost all of them. 

- The disk cache copies remote files to local SSD on the first read. 

- The DBSQL result caches return a whole result set without re-running the query. 

Spark's `.cache()` is rarely needed now that the disk cache handles rescans, and on serverless it is a silent no-op (`is_cached` stays `False`) while `CACHE TABLE` raises outright. 

Before crediting a fast repeat run to your optimization, you need to confirm which cache layer actually answered.

This demo runs a dashboard query cold, then warm, and proves from `system_vw.query.history` that the fast repeat was a result-cache hit, not a faster query. 

You will see why manual caching rituals are retired and what the platform uses in their place.

## Learning Objectives

By the end of this demo, you'll be able to:

1. **Explain what a DataFrame is and when caching helps**: a DataFrame is a plan, not data in memory, and caching pays off only for repeated access.
2. **Describe the cache stack**: disk cache, the DBSQL local and remote result caches and Spark cache.
3. **Explain why manual `.cache()` is retired**, and that it is a no-op on serverless while `CACHE TABLE` raises.
4. **Verify which cache answered** a query using `system_vw.query.history` and the Query Profile.
5. **Benchmark honestly** by turning result reuse off, tying back to Part 2.
6. **Identify what replaced `.cache()`**: the disk cache and the remote result cache, both automatic.

<div style="border-left:4px solid #FFAB00;background:rgba(255,171,0,0.08);padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#B8860B;margin-bottom:6px;font-size:15pt;">Pre-Run Timing Note</strong>
  <div style="color:#0b2026;font-size:14pt;line-height:1.55;">
    This notebook reads from <code>system_vw.query.history</code>, which lags live activity by <strong>5 to 10 minutes</strong>. Run the query cells first, then continue with the explanatory sections. By the time you reach the system-table cells, the rows will have landed. If a cell returns no rows, wait a few minutes and re-run it.
  </div>
</div>

## REQUIRED - SELECT A COMPUTE ENVIRONMENT

<div style="
  border-left: 4px solid #f44336;
  background: #ffebee;
  padding: 14px 18px;
  border-radius: 4px;
  margin: 16px 0;
">
  <strong style="display:block; color:#c62828; margin-bottom:6px; font-size: 1.1em;">Select a Compute Environment</strong>
  <div style="color:#333;">

Before starting this notebook, select the required compute environment listed below.

- **Serverless Compute, Version 5**

![Serverless Select](https://files.training.databricks.com/binder/prod_main/databricks-performance-optimization-en_us-3.0.1/images/20261106T135523Z/Databricks Performance Optimization/Includes/Images/common/select-serverless.png)
<br></br>
  - How to select an environment version:
[AWS](https://docs.databricks.com/aws/en/compute/serverless/dependencies#-select-an-environment-version) |
[Azure](https://learn.microsoft.com/en-us/azure/databricks/compute/serverless/dependencies#-select-an-environment-version) |
[GCP](https://docs.databricks.com/gcp/en/compute/serverless/dependencies#-select-an-environment-version)

**NOTE:**  This notebook requires **Python and was tested using Serverless V5**. The entire demo (both the "before" legacy path and the "after" managed table path) runs on serverless. Other compute may work but is not guaranteed to behave the same or support all features demonstrated.
  </div>
</div>

## A. Classroom Setup

Run the setup below. It builds the shared retail demo tables and defines `DASHBOARD_QUERY_SQL`, the dashboard aggregate we run cold then warm.

In [ ]:
%run ./Includes/Classroom-Setup-05


## B. What a DataFrame Is, and When Caching Helps
<br />
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
</style>

<div style="font-size:14pt;line-height:1.55;color:#0b2026;margin-bottom:18px;">
A DataFrame is a <strong>logical plan</strong>, a named description of a computation over a dataset. It is not a copy of the data held in memory. Because Spark is lazy, calling transformations like <code>.filter()</code> or <code>.groupBy()</code> only builds the plan. Nothing is materialized until an <strong>action</strong> (<code>.count()</code>, <code>.collect()</code>, <code>display()</code>) triggers execution.
</div>

<div style="display:flex;justify-content:center;margin:0 auto 18px;">
<svg viewBox="0 0 820 260" xmlns="http://www.w3.org/2000/svg" style="width:100%;max-width:820px;height:auto;">
  <defs>
    <marker id="df-arr" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#618794"/></marker>
  </defs>
  <!-- Misconception side -->
  <rect x="10" y="10" width="380" height="240" rx="10" fill="#FFF6F4" stroke="#FF5F46" stroke-width="2"/>
  <text x="30" y="40" font-size="13" font-weight="700" fill="#98102A" font-family="sans-serif" letter-spacing="0.5">MISCONCEPTION</text>
  <text x="30" y="66" font-size="14" fill="#0b2026" font-family="sans-serif">"I created a DataFrame, so Spark is</text>
  <text x="30" y="86" font-size="14" fill="#0b2026" font-family="sans-serif">holding that data in memory."</text>
  <!-- Wrong mental model -->
  <rect x="40" y="108" width="130" height="50" rx="6" fill="#FF5F46" fill-opacity="0.15" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="105" y="138" text-anchor="middle" font-size="12" fill="#98102A" font-family="sans-serif" font-weight="600">df = spark.table()</text>
  <line x1="170" y1="133" x2="220" y2="133" stroke="#FF5F46" stroke-width="2" marker-end="url(#df-arr)" style="marker-end:url(#df-arr)"/>
  <rect x="225" y="108" width="140" height="50" rx="6" fill="#FF5F46" fill-opacity="0.15" stroke="#FF5F46" stroke-width="1.5"/>
  <text x="295" y="130" text-anchor="middle" font-size="11" fill="#98102A" font-family="sans-serif" font-weight="600">RAM</text>
  <text x="295" y="148" text-anchor="middle" font-size="11" fill="#98102A" font-family="sans-serif">all rows loaded?</text>
  <!-- X mark -->
  <text x="200" y="200" font-size="28" fill="#FF5F46" font-family="sans-serif" font-weight="700" text-anchor="middle">✗ Not how it works</text>
  <!-- Reality side -->
  <rect x="430" y="10" width="380" height="240" rx="10" fill="#f0faf5" stroke="#00A972" stroke-width="2"/>
  <text x="450" y="40" font-size="13" font-weight="700" fill="#00A972" font-family="sans-serif" letter-spacing="0.5">REALITY</text>
  <text x="450" y="66" font-size="14" fill="#0b2026" font-family="sans-serif">A DataFrame is a plan. Each action</text>
  <text x="450" y="86" font-size="14" fill="#0b2026" font-family="sans-serif">recomputes from the source.</text>
  <!-- Plan box -->
  <rect x="450" y="108" width="120" height="50" rx="6" fill="#00A972" fill-opacity="0.12" stroke="#00A972" stroke-width="1.5"/>
  <text x="510" y="130" text-anchor="middle" font-size="11" fill="#0b2026" font-family="sans-serif" font-weight="600">Logical Plan</text>
  <text x="510" y="148" text-anchor="middle" font-size="11" fill="#618794" font-family="sans-serif">(no data yet)</text>
  <!-- Action arrow -->
  <line x1="570" y1="133" x2="620" y2="133" stroke="#618794" stroke-width="2" marker-end="url(#df-arr)"/>
  <text x="595" y="120" text-anchor="middle" font-size="10" fill="#618794" font-family="sans-serif">action</text>
  <!-- Storage -->
  <rect x="625" y="108" width="160" height="50" rx="6" fill="#4299E0" fill-opacity="0.12" stroke="#4299E0" stroke-width="1.5"/>
  <text x="705" y="130" text-anchor="middle" font-size="11" fill="#0b2026" font-family="sans-serif" font-weight="600">Read from storage</text>
  <text x="705" y="148" text-anchor="middle" font-size="11" fill="#618794" font-family="sans-serif">compute, return result</text>
  <!-- Check mark -->
  <text x="620" y="200" font-size="28" fill="#00A972" font-family="sans-serif" font-weight="700" text-anchor="middle">✓ Each action rescans</text>
  <text x="620" y="226" font-size="12" fill="#618794" font-family="sans-serif" text-anchor="middle">unless something has cached the data</text>
</svg>
</div>

<div style="border-left:6px solid #4299E0;border-radius:8px;padding:18px 24px;background:rgba(66,153,224,0.08);box-shadow:0 2px 8px rgba(27,49,57,0.06);font-size:14pt;line-height:1.55;color:#0b2026;margin-bottom:18px;">
  <strong style="display:block;color:#4299E0;margin-bottom:8px;font-size:15pt;">When caching helps</strong>
  <ul style="margin:0;padding-left:20px;line-height:1.55;">
    <li style="margin-bottom:10px;"><strong>Repeated access to the same data</strong>: iterative ML training that scans the same dataset many times, or interactively exploring the same DataFrame from several angles.</li>
    <li style="margin-bottom:10px;">Without caching, each pass goes back to cloud object storage or recomputes the entire plan. Caching keeps the materialized data close (memory or local disk) so the repeats are fast.</li>
    <li style="margin-bottom:0;"><strong>Single straight-through pipelines</strong> (read, transform, write once) get no benefit, because Spark just streams the computation end to end.</li>
  </ul>
</div>

<div style="border-left:6px solid #00A972;border-radius:8px;padding:18px 24px;background:rgba(0,169,114,0.10);box-shadow:0 2px 8px rgba(27,49,57,0.06);font-size:14pt;line-height:1.55;color:#0b2026;">
  <strong style="display:block;color:#00A972;margin-bottom:8px;font-size:15pt;">On serverless, the platform handles it</strong>
  On serverless you no longer manage caching by hand. The <strong>disk cache</strong> and <strong>result cache</strong> in the next section handle the common repeat-access cases automatically, which is why manual <code>.cache()</code> and <code>.persist()</code> are largely retired here. Understanding the use case still matters so you recognize when repeat access is happening and why the platform's automatic caching helps.
</div>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The misconception in detail
- It is easy to assume that creating a DataFrame, or querying it, means Spark now holds that data in memory. It does not. A DataFrame object is metadata (a logical plan) that describes a series of transformations. The data stays in cloud storage until an action materializes the plan.
- Each action (`.count()`, `.show()`, `display()`, `.write`) triggers a fresh execution from the source unless a cache layer (disk cache, result cache, or explicit Spark cache) intercepts the read.
- This is why re-running the same cell twice can be slow twice: Spark does not remember the previous result unless something caches it.

### When the payoff exists
- Iterative ML workflows (gradient descent over the same feature matrix many passes), interactive EDA that filters, groups, and pivots the same base DataFrame, and any pattern where the same data is scanned more than once within a session.
- For a single linear pipeline (read, join, aggregate, write), caching adds overhead (serialization and storage) with no benefit, because the data flows through exactly once.

### The serverless shift
- On serverless, the disk cache copies remote files to local SSD automatically on first read, so a second scan of the same files is already local. The result cache returns identical query results for about 24 hours without re-execution.
- Together these cover the most common repeat-access scenarios without any code. Manual `.cache()` is retired on serverless (it either no-ops or raises), and the use cases it served are handled by the platform.

</details>


## C. The Cache Stack
<br />
<div class="cw">
<style>
.cw { font-family: sans-serif; max-width: 1100px; margin: 0 auto; color: #0b2026; }
.cw * { box-sizing: border-box; }
.ds-row { display: flex; gap: 24px; align-items: stretch; flex-wrap: wrap; }
.ds-row > * { flex: 1; min-width: 240px; }
.ds-card { background: #F9F7F4; border-radius: 8px; box-shadow: 0 2px 8px rgba(27,49,57,0.06); padding: 20px; border-top: 8px solid var(--accent, #4299E0); }
.blue{--accent:#4299E0;} .green{--accent:#00A972;} .red{--accent:#98102A;}
.ds-card-title { font-size: 18pt; font-weight: 700; line-height: 1.25; margin: 0 0 8px; }
.ds-card-sub { font-size: 14pt; font-weight: 600; color: #1B5162; margin: 0 0 12px; line-height: 1.4; }
.ds-card ul, .ds-card ol { margin: 8px 0 0; padding-left: 20px; font-size: 14pt; line-height: 1.55; color: #0b2026; }
.ds-card li { margin-bottom: 10px; }
.ds-card li:last-child { margin-bottom: 0; }
</style>

<div class="ds-row">
  <div class="ds-card blue">
    <div class="ds-card-title">Disk cache</div>
    <div class="ds-card-sub">Caches DATA. Automatic.</div>
    <ul>
      <li>Copies remote files to local SSD on first read</li>
      <li>On by default on SSD nodes, serverless warehouses, and serverless compute</li>
      <li>Self-invalidates when files change</li>
      <li>Formerly the Delta cache / DBIO cache</li>
    </ul>
  </div>
  <div class="ds-card green">
    <div class="ds-card-title">Result caches (DBSQL)</div>
    <div class="ds-card-sub">Cache whole RESULTS.</div>
    <ul>
      <li><strong>Local</strong>: in-warehouse memory, dies on restart</li>
      <li><strong>Remote</strong>: serverless-only, workspace-shared, ~24h, survives restarts</li>
      <li>Returns the result without re-running the query</li>
    </ul>
  </div>
  <div class="ds-card red">
    <div class="ds-card-title">Spark cache</div>
    <div class="ds-card-sub">Manual. Almost never the answer.</div>
    <ul>
      <li><code>.cache()</code> / <code>.persist()</code>, in-memory blocks</li>
      <li>Lazy, memory-pressured, goes stale, lost on autoscale</li>
      <li>On serverless: DataFrame API is a silent no-op, SQL API raises</li>
    </ul>
  </div>
</div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### The cache stack, top to bottom
- **Disk cache** caches the data files, not results. It copies remote Parquet to local worker SSD the first time a file is read, and serves later reads locally with no code change. It is on by default on SSD-backed nodes, serverless SQL warehouses, and serverless notebook compute, and it detects when files change and invalidates itself. This is why manual Spark caching is rarely needed: rescans are already local.
- **Result caches** cache the entire result set of a query, so an identical query returns instantly without re-running. The local cache lives in the warehouse's memory and dies when the warehouse stops. The remote result cache is serverless-only, is shared across every warehouse in the workspace, survives restarts, and lasts about 24 hours. That remote cache is what drives the cold-versus-warm demo you are about to run.
- **Spark cache** is the manual `.cache()` / `.persist()` you may know from classic Spark. It stores in-memory blocks, is lazy, competes for executor memory, does not auto-invalidate when data changes, and is lost when autoscaling removes a worker. On serverless the DataFrame API (`.cache()`, `.persist()`) accepts the call but does nothing (`is_cached` stays `False`). The SQL API (`CACHE TABLE`, `spark.catalog.cacheTable()`) raises `AnalysisException [NOT_SUPPORTED_WITH_SERVERLESS]`. We cover it so you can recognize and remove it from inherited code.

</details>

## D. Cold vs Warm: The Remote Result Cache

Run the dashboard query twice. The first run is cold: it scans, joins, and aggregates. The repeat is served whole from the remote result cache. Then we prove the repeat was a cache hit from `system_vw.query.history`, not just a faster query.


<div style="border-left:6px solid #00A972;border-radius:8px;padding:18px 24px;background:rgba(0,169,114,0.10);box-shadow:0 2px 8px rgba(27,49,57,0.06);font-size:14pt;line-height:1.55;color:#0b2026;">
  <strong style="display:block;color:#00A972;margin-bottom:8px;font-size:15pt;">Run the query cold then warm on a SQL warehouse</strong>
  <code>benchmark_cold_warm()</code> executes <code>DASHBOARD_QUERY_SQL</code> on a <strong>serverless SQL warehouse</strong> via the SDK Statement Execution API:
  <ul style="margin:8px 0 0;padding-left:20px;line-height:1.55;">
    <li><strong>Cold run</strong>: a unique comment appended to the query forces a result-cache miss. Full scan, join, and aggregate.</li>
    <li><strong>Warm run</strong>: the same query text runs twice. The second call is served whole from the remote result cache.</li>
  </ul>
  <code>use_cached_result</code> is a DBSQL config available only on SQL warehouses, not on serverless notebook compute (Spark Connect). The benchmark targets the warehouse directly so this notebook stays on serverless throughout.
</div>

### The dashboard query
<br />
<div style="max-width:1100px;margin:0 auto;font-family:sans-serif;color:#0b2026;">
  <div style="border-left:4px solid #1B5162;background:#F8F9FC;padding:12px 16px;border-radius:4px;font-size:14pt;line-height:1.55;margin-bottom:12px;">
    <strong>Reference only.</strong> This is the dashboard aggregate that <code>benchmark_cold_warm()</code> runs twice on a serverless SQL warehouse: once cold (cache miss), once warm (result-cache hit).
  </div>
  <pre style="background:#F9F7F4;border:1px solid #EEEDE9;border-radius:8px;padding:16px;overflow-x:auto;margin:0;font-family:Menlo,Consolas,monospace;font-size:13pt;line-height:1.5;color:#0b2026;">SELECT
  p.category,
  c.tier,
  SUM(oi.quantity * p.list_price) AS revenue,
  COUNT(DISTINCT o.order_id)      AS orders
FROM order_items oi
JOIN orders o    ON oi.order_idx = o.order_idx
JOIN products p  ON oi.product_idx = p.product_idx
JOIN customers c ON o.customer_idx = c.customer_idx
GROUP BY p.category, c.tier
ORDER BY revenue DESC</pre>
</div>

In [ ]:
benchmark_cold_warm()

### D1. Prove which cache answered

A faster second run does not confirm your query improved. `system_vw.query.history` marks a result-cache hit with `from_result_cache = TRUE`. 

Note also that a cache-served query has **no Query Profile** at all, so the profile is not where you look for this.


<div style="border-left:6px solid #4299E0;border-radius:8px;padding:18px 24px;background:rgba(66,153,224,0.08);box-shadow:0 2px 8px rgba(27,49,57,0.06);font-size:14pt;line-height:1.55;color:#0b2026;">
  <code>system_vw.query.history</code> records which cache layer answered each query. Look for these columns:
  <table style="margin:12px 0 0;border-collapse:collapse;font-size:14pt;width:100%;">
    <tr style="border-bottom:1px solid #EEEDE9;">
      <td style="padding:8px 12px;font-weight:700;white-space:nowrap;font-size:14pt;"><code>from_result_cache</code></td>
      <td style="padding:8px 12px;font-size:14pt;"><code>TRUE</code> = served from the <strong>result cache</strong> (the warm run). No query re-execution.</td>
    </tr>
    <tr style="border-bottom:1px solid #EEEDE9;">
      <td style="padding:8px 12px;font-weight:700;white-space:nowrap;font-size:14pt;"><code>read_io_cache_percent</code></td>
      <td style="padding:8px 12px;font-size:14pt;">Percentage served from the <strong>disk cache</strong>. This is a different layer. Do not conflate with the result cache.</td>
    </tr>
    <tr style="border-bottom:1px solid #EEEDE9;">
      <td style="padding:8px 12px;font-weight:700;white-space:nowrap;font-size:14pt;"><code>total_duration_ms</code></td>
      <td style="padding:8px 12px;font-size:14pt;">The cost signal. A result-cache hit costs almost nothing.</td>
    </tr>
  </table>
  <div style="margin-top:12px;padding:10px 14px;background:rgba(255,171,0,0.12);border-left:4px solid #FFAB00;border-radius:4px;font-size:14pt;">
    <strong>Note:</strong> <code>system_vw.query.history</code> has a 5 to 10 minute ingestion lag. If the query returns no rows, re-run this cell shortly.
  </div>
</div>

In [ ]:
import time

which_cache_sql = f"""
WITH warm AS (
  SELECT start_time, statement_id, from_result_cache,
         read_io_cache_percent, read_bytes, total_duration_ms
  FROM system_vw.query.history
  WHERE executed_by = current_user()
    AND statement_type = 'SELECT'
    AND start_time > current_timestamp() - INTERVAL 4 HOURS
    AND from_result_cache = true
  ORDER BY start_time DESC
  LIMIT 1
),
cold AS (
  SELECT start_time, statement_id, from_result_cache,
         read_io_cache_percent, read_bytes, total_duration_ms
  FROM system_vw.query.history
  WHERE executed_by = current_user()
    AND statement_type = 'SELECT'
    AND start_time > current_timestamp() - INTERVAL 4 HOURS
    AND from_result_cache = false
    AND read_bytes > 0
    AND start_time < (SELECT start_time FROM warm)
  ORDER BY start_time DESC
  LIMIT 1
)
SELECT 'cold (cache miss)' AS run, * FROM cold
UNION ALL
SELECT 'warm (cache hit)' AS run, * FROM warm
-- no-cache_{int(time.time())}
"""

try:
    df = spark.sql(which_cache_sql)
    if df.count() < 2:
        _show_query_history_fallback("05_cache", my_catalog, DEMO_SCHEMA)
    else:
        df.display()
        print("from_result_cache = true confirms the warm run was served whole from the remote result cache.")
except Exception as e:
    print(f"Could not read system_vw.query.history: {type(e).__name__}: {str(e).splitlines()[0]}")
    print("Re-run once system_vw.query.history access is confirmed in this workspace.")

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Reading the cold-vs-warm result
- The cold run pays the full scan, join, and aggregate. A unique UUID comment appended to the query text forces a result-cache miss (the cache keys on exact statement text). This avoids needing `SET use_cached_result = false`, which is a DBSQL session variable unavailable via the Statement Execution API.
- The warm run returns the whole result set from the remote result cache. It does not re-run the query, which is why it is dramatically faster.
- The proof is `from_result_cache = TRUE` in `system_vw.query.history` for the warm run. Do not trust the clock alone.
- A cache-served query produces no Query Profile. The Query Profile UI will say "Query profile is not available." To force a profile, make a trivial change such as editing or removing a `LIMIT`, which makes the query re-execute.
- Watch the column you read: `from_result_cache` is the result cache. `read_io_cache_percent` is the disk cache, a completely different layer. Crediting the wrong cache is a common mistake.
- The remote result cache is serverless-only and shared across the whole workspace, so a colleague on a different warehouse can get the warm result too, for about 24 hours.

</details>

## E. Spark Cache: Useless on Serverless

The manual `.cache()` you may know from classic Spark is not the tool here. The disk cache already makes rescans local. On serverless the behavior is split: the DataFrame API (`.cache()`, `.persist()`) is a silent no-op, and the SQL API (`CACHE TABLE`) raises outright. Either way, manual caching provides no benefit. The cells below demonstrate both.


<div id="cacheFlashcardRoot"></div>

<script>
(function() {
    var flashcards = [
        {
            category: "DataFrame API",
            categoryColor: "#FFAB00",
            question: "df.cache() on serverless?",
            answer: "<b>Silent no-op.</b>"
        },
        {
            category: "SQL API",
            categoryColor: "#98102A",
            question: "CACHE TABLE on serverless?",
            answer: "<b>Raises AnalysisException.</b>"
        }
    ];

    var flippedCards = {};

    function render() {
        var root = document.getElementById('cacheFlashcardRoot');
        var html = '';

        html += '<div style="font-family: sans-serif; width: 100%; margin: 10px 0; padding: 24px; background: #F9F7F4; border-radius: 12px; border: 1px solid #EEEDE9; box-sizing: border-box;">';

        html += '<div style="display: flex; justify-content: space-between; align-items: center; margin-bottom: 20px;">';
        html += '<div style="font-size: 16pt; font-weight: 700; color: #0b2026;">Spark Cache on Serverless</div>';
        html += '<div style="font-size: 14pt; color: #618794;">Click a card to flip it</div>';
        html += '</div>';

        html += '<div style="display: grid; grid-template-columns: 1fr 1fr; gap: 20px;">';

        flashcards.forEach(function(card, index) {
            var isFlipped = flippedCards[index] === true;

            html += '<div class="cacheFlashcard" data-index="' + index + '" style="cursor: pointer; perspective: 1000px; height: 240px;">';
            html += '<div class="cache-card-inner" style="position: relative; width: 100%; height: 100%; transition: transform 0.6s; transform-style: preserve-3d;' + (isFlipped ? ' transform: rotateY(180deg);' : '') + '">';

            // Front
            html += '<div style="position: absolute; width: 100%; height: 100%; backface-visibility: hidden; -webkit-backface-visibility: hidden; background: #FFFFFF; border: 2px solid #EEEDE9; border-radius: 10px; padding: 20px; box-sizing: border-box; display: flex; flex-direction: column;">';
            html += '<div style="margin-bottom: 12px;"><span style="background: ' + card.categoryColor + '; color: white; padding: 4px 12px; border-radius: 20px; font-size: 14pt; font-weight: 600;">' + card.category + '</span></div>';
            html += '<div style="flex: 1; display: flex; align-items: center; justify-content: center; text-align: center;">';
            html += '<div style="font-size: 14pt; color: #0b2026; line-height: 1.5; font-weight: 500;">' + card.question + '</div>';
            html += '</div>';
            html += '<div style="text-align: center; color: #618794; font-size: 14pt; margin-top: 8px;">Click to reveal</div>';
            html += '</div>';

            // Back
            html += '<div style="position: absolute; width: 100%; height: 100%; backface-visibility: hidden; -webkit-backface-visibility: hidden; transform: rotateY(180deg); background: #FFFFFF; border: 2px solid ' + card.categoryColor + '; border-radius: 10px; padding: 20px; box-sizing: border-box; display: flex; flex-direction: column;">';
            html += '<div style="margin-bottom: 12px;"><span style="background: ' + card.categoryColor + '; color: white; padding: 4px 12px; border-radius: 20px; font-size: 14pt; font-weight: 600;">' + card.category + '</span></div>';
            html += '<div style="flex: 1; display: flex; align-items: center;">';
            html += '<div style="font-size: 14pt; color: #0b2026; line-height: 1.6;">' + card.answer + '</div>';
            html += '</div>';
            html += '<div style="text-align: center; color: #618794; font-size: 14pt; margin-top: 8px;">Click to flip back</div>';
            html += '</div>';

            html += '</div>';
            html += '</div>';
        });

        html += '</div>';

        html += '<div style="margin-top: 20px; padding-top: 16px; border-top: 2px solid #EEEDE9; display: flex; justify-content: space-between; align-items: center;">';
        html += '<div style="color: #618794; font-size: 14pt;">Either way, manual caching is useless on serverless. The disk cache handles rescans.</div>';
        html += '<button id="cacheResetBtn" style="padding: 10px 20px; background: #1B5162; color: white; border: none; border-radius: 6px; cursor: pointer; font-size: 14pt; font-weight: 600;">Reset All</button>';
        html += '</div>';

        html += '</div>';

        root.innerHTML = html;
        attachEvents();
    }

    function attachEvents() {
        var cards = document.querySelectorAll('.cacheFlashcard');
        var resetBtn = document.getElementById('cacheResetBtn');

        cards.forEach(function(card) {
            card.addEventListener('click', function() {
                var index = parseInt(this.dataset.index);
                flippedCards[index] = !flippedCards[index];
                var inner = this.querySelector('.cache-card-inner');
                inner.style.transform = flippedCards[index] ? 'rotateY(180deg)' : 'rotateY(0deg)';
            });
            card.addEventListener('mouseenter', function() {
                this.querySelector('.cache-card-inner').style.boxShadow = '0 8px 24px rgba(27,49,57,0.12)';
            });
            card.addEventListener('mouseleave', function() {
                this.querySelector('.cache-card-inner').style.boxShadow = 'none';
            });
        });

        resetBtn.addEventListener('click', function() {
            flippedCards = {};
            document.querySelectorAll('.cache-card-inner').forEach(function(inner) {
                inner.style.transform = 'rotateY(0deg)';
            });
        });
        resetBtn.addEventListener('mouseenter', function() { this.style.background = '#0b2026'; });
        resetBtn.addEventListener('mouseleave', function() { this.style.background = '#1B5162'; });
    }

    if (typeof NodeList.prototype.forEach !== 'function') {
        NodeList.prototype.forEach = Array.prototype.forEach;
    }

    render();
})();
</script>

In [ ]:
df = spark.sql(DASHBOARD_QUERY_SQL)
df.cache()
df.count()
print(f"is_cached = {df.is_cached}")
df.unpersist()

In [ ]:
try:
    spark.sql(f"CACHE TABLE `{my_catalog}`.`{DEMO_SCHEMA}`.customers")
    print("CACHE TABLE did NOT raise (unexpected on serverless)")
except Exception as e:
    print(f"{type(e).__name__}: {str(e).splitlines()[0]}")

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why Spark cache is retired
- The disk cache already copies remote files to local SSD on first read and serves later reads locally. Manual Spark caching duplicates that, in memory, and adds problems.
- Spark cache is lazy (nothing happens until an action), competes for executor memory, and does not auto-invalidate when the underlying data changes, so it goes stale silently.
- On autoscaling clusters, cached blocks live on workers, so when a worker is decommissioned the cache is lost and partitions are re-read anyway.
- On serverless, the behavior is split. The DataFrame API (`df.cache()`, `df.persist()`) accepts the call but does nothing: `is_cached` stays `False`. The SQL API (`CACHE TABLE`, `spark.catalog.cacheTable()`) raises `AnalysisException [NOT_SUPPORTED_WITH_SERVERLESS]`. Either way, manual caching provides no benefit.
- A subtle point for this audience: Spark Connect support is not the same as serverless support. The DataFrame cache APIs work over Databricks Connect against a classic cluster, but on serverless they are no-ops.
- The disk cache and result cache together cover the two scenarios that motivated manual `.cache()`: re-reading the same files and re-running the same query. Section G summarizes this.

</details>


## F. Cache Warming

<div style="border-left:4px solid #98102A;background:#FDE0DC;padding:14px 18px;border-radius:4px;margin:16px 0;font-family:sans-serif;">
  <strong style="display:block;color:#98102A;margin-bottom:6px;font-size:15pt;">Anti-Patterns</strong>
  <div style="color:#0b2026;font-size:14pt;">
    <ul style="margin:8px 0 0;padding-left:20px;line-height:1.55;">
      <li><strong><code>CACHE SELECT</code> to pre-warm.</strong> Ignored on SQL warehouses and DBR 14.2 and above.</li>
      <li><strong>Manual <code>.cache()</code> / <code>.persist()</code>.</strong> The disk cache covers rescans.</li>
      <li><strong>Benchmarking against a warm cache.</strong> Turn result reuse off with <code>SET use_cached_result = false</code>.</li>
      <li><strong>Sizing clusters around Spark cache memory.</strong> The disk cache uses local SSD, not the memory you pay to over-provision.</li>
    </ul>
  </div>
</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Retiring the cache rituals
- `CACHE SELECT` used to pre-warm the cache. On SQL warehouses and DBR 14.2 and above it is ignored, because an enhanced disk-caching algorithm decides what to keep local. Leave it out of new code.
- Manual `.cache()` and `.persist()` are the rituals to drop. The disk cache already handles the rescans they were meant to speed up, and on serverless they are silent no-ops (`is_cached` stays `False`).
- Do not benchmark against a warm cache. A second run that hits the result cache is not measuring your query. Turn result reuse off for the honest number, then let the cache help in production.
- Do not buy a bigger cluster to hold a Spark cache in memory. The disk cache uses local SSD and is automatic, so that memory sizing is wasted money.

</details>


## G. What Replaced Manual Caching

The platform handles repeat-read scenarios with two automatic caches. Neither requires any action from you:

<div style="font-family:sans-serif;max-width:1100px;margin:0 auto;color:#0b2026;">

<table style="width:100%;border-collapse:collapse;background:#F9F7F4;border-radius:8px;overflow:hidden;box-shadow:0 2px 8px rgba(27,49,57,0.06);">
  <tr>
    <th style="background:#1B5162;color:#fff;text-align:left;padding:14px 18px;font-size:14pt;font-weight:700;">Cache layer</th>
    <th style="background:#1B5162;color:#fff;text-align:left;padding:14px 18px;font-size:14pt;font-weight:700;">What it does</th>
    <th style="background:#1B5162;color:#fff;text-align:left;padding:14px 18px;font-size:14pt;font-weight:700;">What it covers</th>
  </tr>
  <tr>
    <td style="padding:14px 18px;border-bottom:1px solid #EEEDE9;font-size:14pt;line-height:1.5;vertical-align:top;font-weight:600;">Result cache</td>
    <td style="padding:14px 18px;border-bottom:1px solid #EEEDE9;font-size:14pt;line-height:1.5;vertical-align:top;">Returns the whole query result for ~24 hours without re-executing. Workspace-shared on serverless.</td>
    <td style="padding:14px 18px;border-bottom:1px solid #EEEDE9;font-size:14pt;line-height:1.5;vertical-align:top;">Any repeated identical SQL text.</td>
  </tr>
  <tr style="background:#FFFFFF;">
    <td style="padding:14px 18px;font-size:14pt;line-height:1.5;vertical-align:top;font-weight:600;">Disk cache</td>
    <td style="padding:14px 18px;font-size:14pt;line-height:1.5;vertical-align:top;">Copies remote Parquet/Delta files to local SSD on first read. Subsequent reads are local. Auto-invalidates on file change.</td>
    <td style="padding:14px 18px;font-size:14pt;line-height:1.5;vertical-align:top;">Any rescan of the same data files, even if the query changes.</td>
  </tr>
</table>

</div>

##### EXPAND FOR ADDITIONAL DETAILS

<details>

### Why these two caches replace .cache()
- The **result cache** (~24h, workspace-shared, serverless-only) covers the "re-run the same query" case that people used `.cache()` for. If you re-run the exact same SQL text, the platform serves the cached result without re-executing. Prove a hit with `from_result_cache = TRUE` in `system_vw.query.history`.
- The **disk cache** covers the "re-read the same files" case. Once a file is read, subsequent scans come from local SSD regardless of the query text. On by default on serverless, serverless SQL warehouses, and SSD-backed classic nodes. Measured with `read_io_cache_percent`.
- Together they handle the two scenarios that motivated manual `.cache()`: same-query rescans and same-file rescans. No code, no memory management, no stale-data risk.

</details>


## H. Key Takeaways
<br />
<div style="font-family:sans-serif;max-width:1100px;margin:0 auto;color:#0b2026;">

<div style="background:#1B5162;color:#fff;border-radius:10px;padding:22px 28px;text-align:center;border-top:6px solid #FF5F46;box-shadow:0 2px 10px rgba(27,49,57,0.12);margin-bottom:20px;">
  <div style="font-size:17pt;font-weight:700;color:#fff;line-height:1.3;margin-bottom:8px;">Two automatic caches cover the scenarios that motivated manual .cache()</div>
  <div style="font-size:14pt;color:#F9F7F4;font-weight:600;line-height:1.4;">Verify which layer answered before crediting a fast second run to your query change.</div>
</div>

<div style="border-left:6px solid #FFAB00;border-radius:8px;padding:18px 24px;background:rgba(255,171,0,0.12);box-shadow:0 2px 8px rgba(27,49,57,0.06);font-size:14pt;line-height:1.55;color:#0b2026;">
  <div style="font-size:14pt;font-weight:700;letter-spacing:1.5px;text-transform:uppercase;color:#FFAB00;margin-bottom:6px;">What to remember from this demo</div>
  <ul style="margin:0;padding-left:20px;font-size:14pt;line-height:1.55;color:#0b2026;">
    <li style="margin-bottom:10px;">The <strong>disk cache</strong> copies data files to local SSD on first read and serves subsequent reads locally. On by default on serverless, serverless SQL warehouses, and SSD-backed classic nodes.</li>
    <li style="margin-bottom:10px;">The <strong>remote result cache</strong> returns whole query results for ~24 hours without re-executing. Workspace-shared on serverless. Keys on exact statement text.</li>
    <li style="margin-bottom:10px;">Manual <strong>Spark <code style="font-size:14pt;">.cache()</code></strong> is retired. On serverless the DataFrame API is a silent no-op (<code style="font-size:14pt;">is_cached = False</code>). <code style="font-size:14pt;">CACHE TABLE</code> raises <code style="font-size:14pt;">NOT_SUPPORTED_WITH_SERVERLESS</code>.</li>
    <li style="margin-bottom:10px;">Prove a result-cache hit with <code style="font-size:14pt;">from_result_cache = TRUE</code> in <code style="font-size:14pt;">system_vw.query.history</code>. A cache-served query produces <strong>no Query Profile</strong>.</li>
    <li style="margin-bottom:0;">Benchmark honestly: <code style="font-size:14pt;">SET use_cached_result = false</code> disables the result caches so you measure query execution, not cache retrieval.</li>
  </ul>
</div>

</div>

## Conclusion

In this demo, you:

1. Established that a **DataFrame is a logical plan**, not data held in memory, and that caching pays off only for repeated access to the same data.
2. Walked the **cache stack**: disk cache, result caches (local and remote), and Spark cache.
3. Ran a dashboard query **cold then warm** and proved the warm run was a result-cache hit using `from_result_cache = TRUE` in `system_vw.query.history`.
4. Confirmed that manual **Spark `.cache()`** is a silent no-op on serverless (`is_cached = False`) and that `CACHE TABLE` raises outright.
5. Used `SET use_cached_result = false` to disable the result caches and measure query execution without cache interference.
6. Identified the **result cache** and **disk cache** as the two automatic layers that cover the scenarios manual `.cache()` was used for.

Next, in **Lab 1: Fix the Slow Report**, you apply Parts 2 through 5 to a broken report: measure it, diagnose the layout, re-lay it out, and prove the speedup.


&copy; <span id="dbx-year"></span> Databricks, Inc. All rights reserved.
Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>
<script>
  document.getElementById("dbx-year").textContent = new Date().getFullYear();
</script>